In [ ]:
# ============================================================================
# AP-1000 FULL CORE CRITICALITY MODEL
# Realistic 157-assembly core with water reflector and enrichment zones
# Author: Eric Yokie | OpenMC v0.15.2
# ============================================================================

import numpy as np
import openmc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patches import Circle
import os

# Pin working directory
os.chdir('/home/eric/Documents/openMC_models/AP-1000')
print(f"Working directory: {os.getcwd()}")

openmc.config['cross_sections'] = '/home/eric/openmc/nuclear_data/endf-b8.0-hdf5/endfb-viii.0-hdf5/cross_sections.xml'

# ============================================================================
# 1. MATERIALS
# Three enrichment zones for realistic low-leakage loading pattern:
#   Zone 1 (outer):  4.45 w/o  — fresh high enrichment
#   Zone 2 (middle): 3.40 w/o  — fresh mid enrichment
#   Zone 3 (inner):  2.35 w/o  — once-burned or low enrichment
# ============================================================================

def make_fuel(enrichment, name):
    fuel = openmc.Material(name=name)
    fuel.add_element('U', 1.0, enrichment=enrichment)
    fuel.add_element('O', 2.0)
    fuel.set_density('g/cm3', 10.40)  # slightly reduced for porosity
    return fuel

fuel1 = make_fuel(2.35, "UO2 2.35 w/o")
fuel1.temperature = 600.0   # K — fuel pellet at HZP

fuel2 = make_fuel(3.40, "UO2 3.40 w/o")
fuel2.temperature = 600.0

fuel3 = make_fuel(4.45, "UO2 4.45 w/o")
fuel3.temperature = 600.0

cladding = openmc.Material(name="Zircaloy-4")
cladding.add_element('Zr', 0.98,   percent_type='wo')
cladding.add_element('Sn', 0.015,  percent_type='wo')
cladding.add_element('Fe', 0.0021, percent_type='wo')
cladding.add_element('Cr', 0.001,  percent_type='wo')
cladding.set_density('g/cm3', 6.55)

# Coolant 
coolant = openmc.Material(name="Coolant (3000 ppm B)")
coolant.add_element('H', 0.111,   percent_type='wo')
coolant.add_element('O', 0.887,   percent_type='wo')
coolant.add_element('B', 0.00300, percent_type='wo')
coolant.set_density('g/cm3', 0.7070)
coolant.add_s_alpha_beta('c_H_in_H2O')

# Reflector — boron added here, to the reflector object
reflector = openmc.Material(name="Water Reflector")
reflector.add_element('H', 0.111,   percent_type='wo')
reflector.add_element('O', 0.887,   percent_type='wo')
reflector.add_element('B', 0.00300, percent_type='wo')
reflector.set_density('g/cm3', 0.7070)
reflector.add_s_alpha_beta('c_H_in_H2O')

he_gap = openmc.Material(name="Helium")
he_gap.add_element('He', 1.0)
he_gap.set_density('g/cm3', 0.000178)
he_gap.temperature = 600.0

cladding.temperature = 600.0  # K — cladding at coolant temp
coolant.temperature  = 600.0  # K — bulk coolant HZP
reflector.temperature = 600.0 # K — reflector same as coolant

materials = openmc.Materials([fuel1, fuel2, fuel3, cladding, coolant, he_gap, reflector])
materials.export_to_xml()
print("Materials created successfully.")

# ============================================================================
# 2. PIN-LEVEL GEOMETRY
#      Each call to make_pin_universe() creates its own fresh surface objects.
#      Sharing XPlane/YPlane instances across universes placed in a lattice
#      causes lost-particle errors at pin boundaries.
# ============================================================================

pitch      = 1.2604   # cm — pin pitch
half_pitch = pitch / 2
fuel_r     = 0.4096   # cm — fuel pellet radius
gap_r      = 0.4178   # cm — He gap outer radius
clad_r     = 0.4750   # cm — cladding outer radius
gt_inner_r = 0.5610   # cm — guide tube inner radius
gt_outer_r = 0.6020   # cm — guide tube outer radius

def make_pin_universe(fuel_mat):
    # LESSON LEARNED: fresh surfaces every call — don't share across universes in a lattice
    s_fuel = openmc.ZCylinder(r=fuel_r)
    s_gap  = openmc.ZCylinder(r=gap_r)
    s_clad = openmc.ZCylinder(r=clad_r)
    xm = openmc.XPlane(-half_pitch)
    xp = openmc.XPlane( half_pitch)
    ym = openmc.YPlane(-half_pitch)
    yp = openmc.YPlane( half_pitch)
    cell_box = +xm & -xp & +ym & -yp

    c_fuel    = openmc.Cell(region=-s_fuel,            fill=fuel_mat)
    c_gap     = openmc.Cell(region=+s_fuel & -s_gap, fill=he_gap)
    c_clad    = openmc.Cell(region=+s_gap  & -s_clad,  fill=cladding)
    c_coolant = openmc.Cell(region=+s_clad & cell_box, fill=coolant)
    return openmc.Universe(cells=[c_fuel, c_gap, c_clad, c_coolant])

pin1 = make_pin_universe(fuel1)
pin2 = make_pin_universe(fuel2)
pin3 = make_pin_universe(fuel3)

# Guide tube universe also gets its own independent bounding planes
s_gt_i = openmc.ZCylinder(r=gt_inner_r)
s_gt_o = openmc.ZCylinder(r=gt_outer_r)
gt_xm  = openmc.XPlane(-half_pitch)
gt_xp  = openmc.XPlane( half_pitch)
gt_ym  = openmc.YPlane(-half_pitch)
gt_yp  = openmc.YPlane( half_pitch)
gt_box = +gt_xm & -gt_xp & +gt_ym & -gt_yp

c_gt_water      = openmc.Cell(region=-s_gt_i,            fill=coolant)
c_gt_tube       = openmc.Cell(region=+s_gt_i & -s_gt_o,  fill=cladding)
c_gt_cool       = openmc.Cell(region=+s_gt_o & gt_box,   fill=coolant)
guide_tube_univ = openmc.Universe(cells=[c_gt_water, c_gt_tube, c_gt_cool])

# ============================================================================
# 3. AP-1000 17x17 GUIDE TUBE MAP
# Full 24 guide tubes + 1 instrument tube (center)
# ============================================================================

# 1 = fuel pin, 0 = guide/instrument tube
GT = 0
FP = 1

lattice_map = np.array([
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 0
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 1
    [FP,FP,FP,FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP,FP,FP,FP],  # row 2
    [FP,FP,FP,GT,FP,FP,FP,FP,FP,FP,FP,FP,FP,GT,FP,FP,FP],  # row 3
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 4
    [FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP],  # row 5
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 6
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 7
    [FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP],  # row 8  (center)
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 9
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 10
    [FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP],  # row 11
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 12
    [FP,FP,FP,GT,FP,FP,FP,FP,FP,FP,FP,FP,FP,GT,FP,FP,FP],  # row 13
    [FP,FP,FP,FP,FP,GT,FP,FP,GT,FP,FP,GT,FP,FP,FP,FP,FP],  # row 14
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 15
    [FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP,FP],  # row 16
])

def make_assembly_lattice(pin_univ):
    aw  = pitch * 17
    lat = openmc.RectLattice(name=f"17x17 ({pin_univ.name if hasattr(pin_univ, 'name') else ''})")
    lat.pitch      = (pitch, pitch)
    lat.lower_left = (-aw/2, -aw/2)
    rows = []
    for i in range(17):
        row = []
        for j in range(17):
            row.append(pin_univ if lattice_map[i, j] == FP else guide_tube_univ)
        rows.append(row)
    lat.universes = rows
    return lat

lat1 = make_assembly_lattice(pin1)
lat2 = make_assembly_lattice(pin2)
lat3 = make_assembly_lattice(pin3)

assembly_width = pitch * 17
fuel_height    = 426.72

def make_assembly_universe(lattice, name):
    # Lateral planes: fresh per call (each universe in the core lattice)
    x_lo = openmc.XPlane(-assembly_width / 2)
    x_hi = openmc.XPlane( assembly_width / 2)
    y_lo = openmc.YPlane(-assembly_width / 2)
    y_hi = openmc.YPlane( assembly_width / 2)

    fuel_cell = openmc.Cell(
        region=+x_lo & -x_hi & +y_lo & -y_hi,
        fill=lattice,
        name=name
    )
    return openmc.Universe(name=name, cells=[fuel_cell])

asm1 = make_assembly_universe(lat1, "Assembly Zone 1 (2.35%)")
asm2 = make_assembly_universe(lat2, "Assembly Zone 2 (3.40%)")
asm3 = make_assembly_universe(lat3, "Assembly Zone 3 (4.45%)")

# ============================================================================
# 4. AP-1000 CORE LOADING PATTERN
# 157 fuel assemblies in a circular pattern
# Zone layout (after zone_map swap for low-leakage loading):
#   1 = positions mapped to 4.45 w/o (outer ring in core_map - high enrichment)
#   2 = positions mapped to 3.40 w/o (middle ring)
#   3 = positions mapped to 2.35 w/o (inner region in core_map - low enrichment)
# ============================================================================

core_map = np.array([
    [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0],
    [0,0,0,0,0,1,1,2,2,2,1,1,0,0,0,0,0],
    [0,0,0,0,1,2,2,2,2,2,2,2,1,0,0,0,0],
    [0,0,0,1,2,2,2,2,3,2,2,2,2,1,0,0,0],
    [0,0,1,2,2,2,3,3,3,3,3,2,2,2,1,0,0],
    [0,0,1,2,2,3,3,3,3,3,3,3,2,2,1,0,0],
    [0,1,2,2,2,3,3,3,3,3,3,3,2,2,2,1,0],
    [0,1,2,2,3,3,3,3,3,3,3,3,3,2,2,1,0],
    [0,1,2,2,2,3,3,3,3,3,3,3,2,2,2,1,0],
    [0,0,1,2,2,3,3,3,3,3,3,3,2,2,1,0,0],
    [0,0,1,2,2,2,3,3,3,3,3,2,2,2,1,0,0],
    [0,0,0,1,2,2,2,2,3,2,2,2,2,1,0,0,0],
    [0,0,0,0,1,2,2,2,2,2,2,2,1,0,0,0,0],
    [0,0,0,0,0,1,1,2,2,2,1,1,0,0,0,0,0],
    [0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
])

# ============================================================================
# 5. FULL CORE GEOMETRY
# ============================================================================

core_width  = assembly_width * 17   # full square extent ~363.3 cm
core_r      = core_width / 2
reflector_r = core_r + 20.0         # 20 cm radial reflector

# Water reflector universe — infinite, used for lattice positions outside core
refl_cell = openmc.Cell(fill=reflector, name="Reflector")
refl_univ = openmc.Universe(cells=[refl_cell])

# Build core lattice (17x17 assembly grid)
core_lattice            = openmc.RectLattice(name="AP-1000 Core")
core_lattice.pitch      = (assembly_width, assembly_width)
core_lattice.lower_left = (-core_width/2, -core_width/2)

# higher enrichment is outer
zone_map = {1: asm3, 2: asm2, 3: asm1, 0: refl_univ}
core_universes = []
for i in range(17):
    row = []
    for j in range(17):
        row.append(zone_map[core_map[16 - i, j]])  # flip row index
    core_universes.append(row)
core_lattice.universes = core_universes
core_lattice.outer = refl_univ

print(f"core_lattice center universe: {core_lattice.universes[8][8].name}")

# Core bounding surfaces
z_lo_core = openmc.ZPlane(-fuel_height/2 - 30, boundary_type='vacuum')
z_hi_core = openmc.ZPlane( fuel_height/2 + 30, boundary_type='vacuum')
r_core    = openmc.ZCylinder(r=reflector_r,     boundary_type='vacuum')

# Core fill cell — bounded only by cylinder and axial vacuum planes
# The lattice handles lateral containment internally
core_fill_cell = openmc.Cell(
    region=-r_core & +z_lo_core & -z_hi_core,
    fill=core_lattice,
    name="Core Lattice Region"
)

# Radial reflector — outside cylinder is vacuum, so just need infinite fill
# for the refl_univ lattice positions to handle reflector outside active core
radial_refl_cell = openmc.Cell(
    region=~(-r_core & +z_lo_core & -z_hi_core),
    fill=reflector,
    name="Radial Reflector"
)

root_universe = openmc.Universe(cells=[core_fill_cell, radial_refl_cell])
geometry = openmc.Geometry(root_universe)
geometry.export_to_xml()
print("Geometry created successfully.")

# ============================================================================
# 6. SETTINGS
# ============================================================================

settings = openmc.Settings()
settings.temperature = {
    'default': 600.0,
    'method': 'nearest'
}
settings.batches   = 200
settings.inactive  = 70
settings.particles = 50000
settings.statepoint = {'batches': [200]}
settings.source = openmc.IndependentSource(
    space=openmc.stats.Box(
        (-core_width/2, -core_width/2, -fuel_height/2),
        ( core_width/2,  core_width/2,  fuel_height/2),
        only_fissionable=True
    )
)

settings.export_to_xml()
print("Settings created successfully.")

# ============================================================================
# 7. TALLIES
# ============================================================================

# Assembly-level power map (17x17 mesh over core)
core_mesh           = openmc.RegularMesh(name="core_mesh")
core_mesh.dimension = [17,17]
core_mesh.lower_left  = (-core_width/2, -core_width/2)
core_mesh.upper_right = ( core_width/2,  core_width/2)

core_mesh_filter    = openmc.MeshFilter(core_mesh)
power_tally         = openmc.Tally(name='assembly_power')
power_tally.filters = [core_mesh_filter]
power_tally.scores  = ['fission']

# Fine pin-level mesh over center 3x3 assemblies
pin_mesh           = openmc.RegularMesh(name="pin_mesh")
pin_mesh.dimension = [51, 51]   # 3 assemblies × 17 pins
pin_mesh.lower_left  = (-assembly_width * 1.5, -assembly_width * 1.5)
pin_mesh.upper_right = ( assembly_width * 1.5,  assembly_width * 1.5)

pin_mesh_filter    = openmc.MeshFilter(pin_mesh)
pin_tally          = openmc.Tally(name='pin_power_center')
pin_tally.filters  = [pin_mesh_filter]
pin_tally.scores   = ['fission']

# Fine mesh flux tally over full core
flux_mesh = openmc.RegularMesh(name="flux_mesh")
flux_mesh.dimension   = [170, 170]  # 10 mesh cells per assembly
flux_mesh.lower_left  = (-core_width/2, -core_width/2)
flux_mesh.upper_right = ( core_width/2,  core_width/2)

flux_mesh_filter = openmc.MeshFilter(flux_mesh)
flux_tally = openmc.Tally(name='flux_map')
flux_tally.filters = [flux_mesh_filter]
flux_tally.scores  = ['flux']

# Total flux tally
flux_mesh_coarse = openmc.RegularMesh(name="flux_mesh_coarse")
flux_mesh_coarse.dimension = [68, 68]
flux_mesh_coarse.lower_left  = (-core_width/2, -core_width/2)
flux_mesh_coarse.upper_right = ( core_width/2,  core_width/2)

flux_mesh_coarse_filter = openmc.MeshFilter(flux_mesh_coarse)
energy_filter = openmc.EnergyFilter([0, 0.625, 100e3, 20e6])

flux_energy_tally = openmc.Tally(name='flux_energy')
flux_energy_tally.filters = [flux_mesh_coarse_filter, energy_filter]
flux_energy_tally.scores  = ['flux']

tallies = openmc.Tallies([power_tally, pin_tally, flux_tally, flux_energy_tally])
tallies.export_to_xml()

print("Tallies created successfully.")

# ============================================================================
# 8. GEOMETRY PLOTS
# ============================================================================

plot_xy          = openmc.Plot(name='core_xy')
plot_xy.basis    = 'xy'
plot_xy.origin   = (0, 0, 0)
plot_xy.width    = (core_width * 1.05, core_width * 1.05)
plot_xy.pixels   = (2000, 2000)
plot_xy.color_by = 'material'
plot_xy.colors   = {
    fuel1:     (255,  50,  50),   # red    — 2.35%
    fuel2:     (255, 165,   0),   # orange — 3.40%
    fuel3:     (255, 255,  50),   # yellow — 4.45%
    cladding:  (180, 180, 180),   # grey
    coolant:   ( 30, 100, 220),   # blue
    reflector: ( 10,  60, 180),   # dark blue
}

plot_xz          = openmc.Plot(name='core_xz')
plot_xz.basis    = 'xz'
plot_xz.origin   = (0, 0, 0)
plot_xz.width    = (core_width * 1.05, fuel_height * 1.2)
plot_xz.pixels   = (2000, 1000)
plot_xz.color_by = 'material'
plot_xz.colors   = plot_xy.colors

# Single assembly — individual pins clearly visible (~100 px/pin)
plot_asm_pins          = openmc.Plot(name='assembly_pins')
plot_asm_pins.basis    = 'xy'
plot_asm_pins.origin   = (0, 0, 0)
plot_asm_pins.width    = (assembly_width, assembly_width)
plot_asm_pins.pixels   = (1700, 1700)
plot_asm_pins.color_by = 'material'
plot_asm_pins.colors   = plot_xy.colors

# 3×3 assembly cluster — pin-resolved
plot_3x3_pins          = openmc.Plot(name='3x3_pins')
plot_3x3_pins.basis    = 'xy'
plot_3x3_pins.origin   = (0, 0, 0)
plot_3x3_pins.width    = (assembly_width * 3, assembly_width * 3)
plot_3x3_pins.pixels   = (3000, 3000)
plot_3x3_pins.color_by = 'material'
plot_3x3_pins.colors   = plot_xy.colors

plots = openmc.Plots([plot_xy, plot_xz, plot_asm_pins, plot_3x3_pins])
plots.export_to_xml()
print("Plot XML created.")


# ============================================================================
# 9. RUN
# ============================================================================

# Delete old statepoints only
for f in os.listdir('.'):
    if f.endswith('.h5') or f == 'tallies.out':
        os.remove(f)

print("\n" + "="*60)
print("  AP-1000 FULL CORE — OpenMC Monte Carlo Simulation")
print(f"  157 assemblies | 3 enrichment zones | radial reflector")
print(f"  {settings.particles:,} particles/batch | {settings.batches} batches")
print("="*60 + "\n")

# Verify the lattice is actually using the flipped map
print(f"\ncore_lattice center universe: {core_lattice.universes[8][8].name}")

openmc.run()

# ============================================================================
# 10. RESULTS
# ============================================================================

# sp = openmc.StatePoint('statepoint.150.h5')

sp = openmc.StatePoint('statepoint.200.h5')

keff     = sp.keff.nominal_value
keff_std = sp.keff.std_dev
rho_pcm  = (keff - 1.0) / keff * 1e5
status   = "SUPERCRITICAL" if keff > 1.0 else "SUBCRITICAL"

print("\n" + "="*60)
print("  CRITICALITY RESULTS — AP-1000 Full Core")
print("="*60)
print(f"  k-eff        = {keff:.5f} ± {keff_std:.5f}")
print(f"  Reactivity   = {rho_pcm:+.1f} pcm")
print(f"  Status       = {status}")
print("="*60 + "\n")


In [ ]:
# ============================================================================
# 11. VISUALIZATION — Portfolio Quality
# ============================================================================
asm_tally  = sp.get_tally(name='assembly_power')
asm_power  = asm_tally.get_values(scores=['fission']).reshape(17, 17)
asm_power  = asm_power[::-1]

pin_tally  = sp.get_tally(name='pin_power_center')
pin_power  = pin_tally.get_values(scores=['fission']).reshape(51, 51)

# Normalize: mask reflector positions
asm_power_plot = asm_power.copy()
for i in range(17):
    for j in range(17):
        if core_map[i, j] == 0:
            asm_power_plot[i, j] = np.nan

asm_mean       = np.nanmean(asm_power_plot)
asm_power_norm = asm_power_plot / asm_mean

# ============================================================================
# FIGURE: IEEE publication style
# ============================================================================

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':         12,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'axes.linewidth':    0.8,
    'axes.edgecolor':    '#333333',
    'axes.facecolor':    '#f9f9f9',
    'figure.facecolor':  'white',
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction':   'out',
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'ytick.direction':   'out',
    'xtick.top':         False,
    'ytick.right':       False,
    'legend.frameon':    True,
    'legend.edgecolor':  '#cccccc',
    'legend.facecolor':  'white',
    'legend.fontsize':   10,
    'lines.linewidth':   1.2,
    'grid.linewidth':    0.5,
    'grid.color':        '#dddddd',
    'grid.linestyle':    '-',
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
})

fig = plt.figure(figsize=(15, 8))  # down from (18, 12)

# fig.suptitle(
#     'AP-1000 Pressurized Water Reactor — Full Core Monte Carlo Criticality Analysis',
#     fontsize=12, fontweight='bold', y=0.98
# )

# info = (f"OpenMC v0.15.2  |  157 Fuel Assemblies  |  17$\\times$17 Pin Arrays  |  "
#         f"3 Enrichment Zones  |  Radial Water Reflector  |  "
#         f"$k_{{\\mathrm{{eff}}}}$ = {keff:.5f} $\\pm$ {keff_std:.5f}  |  "
#         f"$\\rho$ = {rho_pcm:+.0f} pcm")
# fig.text(0.5, 0.945, info, ha='center', fontsize=8.5, style='italic')

gs = fig.add_gridspec(2, 3,
                      left=0.05, right=0.97,
                      top=0.92, bottom=0.07,
                      hspace=0.55,
                      wspace=0.45)

ax_keff   = fig.add_subplot(gs[0, 0])
ax_power  = fig.add_subplot(gs[0, 1])
ax_zone   = fig.add_subplot(gs[0, 2])
ax_radial = fig.add_subplot(gs[1, 0])   # radial profile bottom left
ax_pin    = fig.add_subplot(gs[1, 1])   # pin detail bottom center
                                         # gs[1,2] intentionally empty

# --- 1. k-eff Convergence ---
keffs   = sp.k_generation
batches = np.arange(1, len(keffs) + 1)
active  = batches[settings.inactive:]
keffs_a = keffs[settings.inactive:]

ax_keff.plot(batches[:settings.inactive], keffs[:settings.inactive],
             color='gray', lw=0.8, alpha=0.7, label='Inactive')
ax_keff.plot(active, keffs_a, color='black', lw=0.9, alpha=0.8, label='Active')
ax_keff.axhline(keff, color='black', ls='--', lw=1.2,
                label=f'$k_{{\\mathrm{{eff}}}}$ = {keff:.5f}')
ax_keff.fill_between(active, keff - keff_std, keff + keff_std,
                     alpha=0.20, color='gray', label=f'$\\pm 1\\sigma$')
ax_keff.axvline(settings.inactive, color='gray', ls=':', lw=0.8)
ax_keff.set_xlabel('Generation')
ax_keff.set_ylabel('$k_{\\mathrm{eff}}$')
ax_keff.legend(fontsize=9)
ax_keff.grid(True)
ax_keff.set_title('(a) $k_{\\mathrm{eff}}$ Convergence')
ax_keff.minorticks_on()

# --- 2. Assembly Power Map ---
cmap_power = plt.cm.hot_r.copy()
cmap_power.set_bad(color='white')

im_power = ax_power.imshow(asm_power_norm, cmap=cmap_power,
                            origin='lower', vmin=0.6, vmax=1.4,
                            extent=[-core_width/2, core_width/2,
                                    -core_width/2, core_width/2])
cb = plt.colorbar(im_power, ax=ax_power, fraction=0.046, pad=0.04)
cb.set_label('Normalized Power Density')
cb.ax.tick_params(labelsize=7, direction='in')

for i in range(17):
    for j in range(17):
        if core_map[i, j] > 0 and not np.isnan(asm_power_norm[i, j]):
            x   = -core_width/2 + (j + 0.5) * assembly_width
            y   = -core_width/2 + (i + 0.5) * assembly_width
            val = asm_power_norm[i, j]
            col = 'white' if val > 1.15 else 'black'
            ax_power.text(x, y, f'{val:.2f}', ha='center', va='center',
                         fontsize=4.0, color=col)

ax_power.set_xlabel('$x$ (cm)')
ax_power.set_ylabel('$y$ (cm)')
ax_power.set_title('(b) Assembly Power Distribution')
ax_power.minorticks_on()

# --- 3. Enrichment Zone Map ---
cmap_zone   = mcolors.ListedColormap(['white', '#d9e8f5', '#6baed6', '#2171b5'])
bounds_zone = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm_zone   = mcolors.BoundaryNorm(bounds_zone, cmap_zone.N)

im_zone = ax_zone.imshow(core_map, cmap=cmap_zone, norm=norm_zone,
                          origin='lower',
                          extent=[-core_width/2, core_width/2,
                                  -core_width/2, core_width/2])

theta = np.linspace(0, 2*np.pi, 300)
ax_zone.plot(reflector_r * np.cos(theta), reflector_r * np.sin(theta),
             'k--', lw=0.8, alpha=0.6)

legend_patches = [
    mpatches.Patch(facecolor='white',   edgecolor='black', label='Reflector'),
    mpatches.Patch(facecolor='#d9e8f5', edgecolor='black', label='4.45 w/o — Outer (49 asm.)'),
    mpatches.Patch(facecolor='#6baed6', edgecolor='black', label='3.40 w/o — Middle (72 asm.)'),
    mpatches.Patch(facecolor='#2171b5', edgecolor='black', label='2.35 w/o — Inner (36 asm.)'),
]
ax_zone.legend(handles=legend_patches,
               loc='upper center',
               bbox_to_anchor=(0.5, -0.2),
               ncol=2,
               borderaxespad=0,
               fontsize=11)
ax_zone.set_xlabel('$x$ (cm)')
ax_zone.set_ylabel('$y$ (cm)')
ax_zone.set_title('(c) Core Loading Pattern')
ax_zone.minorticks_on()

# --- 4. Pin Power Detail ---
cmap_pin = plt.cm.hot_r.copy()
cmap_pin.set_bad(color='white')

im_pin = ax_pin.imshow(pin_power, cmap=cmap_pin, origin='lower',
                        extent=[-assembly_width*1.5, assembly_width*1.5,
                                -assembly_width*1.5, assembly_width*1.5])
cb2 = plt.colorbar(im_pin, ax=ax_pin, fraction=0.046, pad=0.04)
cb2.set_label('Fission Rate (arb. units)')
cb2.ax.tick_params(labelsize=7, direction='in')

for k in range(-1, 2):
    ax_pin.axvline(k * assembly_width, color='black', lw=0.6, alpha=0.6)
    ax_pin.axhline(k * assembly_width, color='black', lw=0.6, alpha=0.6)
ax_pin.set_xlabel('$x$ (cm)')
ax_pin.set_ylabel('$y$ (cm)')
ax_pin.set_title('(d) Pin Power Detail')
ax_pin.minorticks_on()

# --- 5. Radial Power Profile ---
radial_power = []
radial_dist  = []
cx, cy = 8, 8
for i in range(17):
    for j in range(17):
        if core_map[i, j] > 0:
            r = np.sqrt((i - cx)**2 + (j - cy)**2) * assembly_width
            radial_dist.append(r)
            radial_power.append(asm_power_norm[i, j])

radial_dist  = np.array(radial_dist)
radial_power = np.array(radial_power)

ax_radial.scatter(radial_dist, radial_power,
                  facecolors='none', edgecolors='black',
                  s=20, lw=0.6, zorder=3, label='Assembly')

bins        = np.linspace(0, radial_dist.max() + 1, 10)
bin_means   = []
bin_centers = []
for b in range(len(bins) - 1):
    mask = (radial_dist >= bins[b]) & (radial_dist < bins[b+1])
    if mask.sum() > 0:
        bin_means.append(radial_power[mask].mean())
        bin_centers.append((bins[b] + bins[b+1]) / 2)

ax_radial.plot(bin_centers, bin_means, color='black',
               lw=1.2, marker='s', markersize=4, label='Bin average', zorder=4)
ax_radial.axhline(1.0, color='gray', ls='--', lw=0.8)
ax_radial.set_xlabel('Distance from Core Center (cm)')
ax_radial.set_ylabel('Normalized Power Density')
ax_radial.legend()
ax_radial.grid(True)
ax_radial.set_title('(e) Radial Power Profile')
ax_radial.minorticks_on()

ax_pin.set_aspect('equal')

plt.savefig('ap1000_full_core_results.pdf', bbox_inches='tight', facecolor='white')

plt.show()
print("\nFigure saved: ap1000_full_core_results.pdf")
print("Full core simulation complete")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# Generate the geometry plots first
plot_core = openmc.Plot(name='core_xy')
plot_core.basis    = 'xy'
plot_core.origin   = (0, 0, 0)
plot_core.width    = (core_width * 1.1, core_width * 1.1)
plot_core.pixels   = (3000, 3000)
plot_core.color_by = 'material'
plot_core.colors   = {
    fuel1:     (220, 50,  50),
    fuel2:     (240, 140,  0),
    fuel3:     (255, 210,  0),
    cladding:  (160, 160, 160),
    coolant:   ( 70, 130, 200),
    reflector: ( 30,  80, 160),
}

plot_asm = openmc.Plot(name='assembly_xy')
plot_asm.basis    = 'xy'
plot_asm.origin   = (0, 0, 0)
plot_asm.width    = (assembly_width * 3, assembly_width * 3)
plot_asm.pixels   = (3000, 3000)
plot_asm.color_by = 'material'
plot_asm.colors   = plot_core.colors

plot_xz = openmc.Plot(name='core_xz')
plot_xz.basis    = 'xz'
plot_xz.origin   = (0, 0, 0)
plot_xz.width    = (core_width * 1.1, fuel_height * 1.3)
plot_xz.pixels   = (3000, 1500)
plot_xz.color_by = 'material'
plot_xz.colors   = plot_core.colors

plots = openmc.Plots([plot_core, plot_asm, plot_xz])
plots.export_to_xml()
openmc.plot_geometry()

# Now load and display with annotations
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor('white')

fig.suptitle('AP-1000 Pressurized Water Reactor — Core Geometry',
             fontsize=14, fontweight='bold', y=1.01)

# --- Plot 1: Full core XY ---
img1 = Image.open('plot_5.png')
axes[0].imshow(img1)
axes[0].set_title('Full Core — Top View', fontsize=11, fontweight='bold', pad=10)
axes[0].axis('off')

# Annotate radial zones
axes[0].text(0.5, -0.04, 'Diameter ≈ 364 cm active core + 20 cm reflector',
             transform=axes[0].transAxes, ha='center', fontsize=8, style='italic')

# --- Plot 2: 3x3 assembly pin detail ---
img2 = Image.open('plot_6.png')
axes[1].imshow(img2)
axes[1].set_title('Center 3x3 Assemblies — Pin Detail', fontsize=11, fontweight='bold', pad=10)
axes[1].axis('off')
axes[1].text(0.5, -0.04, '17x17 pin array per assembly | pitch = 1.26 cm',
             transform=axes[1].transAxes, ha='center', fontsize=8, style='italic')

# --- Plot 3: Axial cross section ---
img3 = Image.open('plot_7.png')
axes[2].imshow(img3)
axes[2].set_title('Full Core — Side View', fontsize=11, fontweight='bold', pad=10)
axes[2].axis('off')
axes[2].text(0.5, -0.04, 'Active fuel height = 426.72 cm | 30 cm axial reflector',
             transform=axes[2].transAxes, ha='center', fontsize=8, style='italic')

# --- Shared legend ---
# In your geometry plot legend
legend_patches = [
    mpatches.Patch(color=(255/255,  50/255,  50/255), label='2.35 w/o — Inner'),
    mpatches.Patch(color=(255/255, 165/255,   0/255), label='3.40 w/o — Middle'),
    mpatches.Patch(color=(255/255, 255/255,  50/255), label='4.45 w/o — Outer'),
    mpatches.Patch(color=( 30/255, 100/255, 220/255), label='Borated Coolant (H₂O + B)'),
    mpatches.Patch(color=( 10/255,  60/255, 180/255), label='Water Reflector'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           fontsize=9, frameon=True, edgecolor='gray',
           bbox_to_anchor=(0.5, -0.06))

plt.tight_layout()
plt.savefig('ap1000_geometry_poster.png', dpi=200, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✓ Geometry poster saved: ap1000_geometry_poster.png")

In [ ]:
# ============================================================================
# BORON LETDOWN SWEEP
# Run OpenMC at several boron concentrations and interpolate CBC
# ============================================================================

boron_ppm  = [0, 500, 1000, 1500, 2000, 2500, 3000]
keff_vals  = []
keff_stds  = []

for ppm in boron_ppm:
    # Convert ppm to weight fraction
    # ppm is mg boron / kg solution ≈ g boron / 1e6 g solution
    wo_boron = ppm / 1e6

    # Rebuild coolant and reflector with new boron concentration
    # H and O must sum to (1 - wo_boron)
    wo_water = 1.0 - wo_boron
    wo_H = 0.111 * wo_water
    wo_O = 0.889 * wo_water

    for mat in [coolant, reflector]:
        mat.nuclides.clear()
        mat.add_element('H', wo_H,    percent_type='wo')
        mat.add_element('O', wo_O,    percent_type='wo')
        if ppm > 0:
            mat.add_element('B', wo_boron, percent_type='wo')

    # Re-export and run
    for f in os.listdir('.'):
        if f.endswith('.h5') or f.endswith('.xml') or f == 'tallies.out':
            os.remove(f)

    materials.export_to_xml()
    geometry.export_to_xml()
    settings.export_to_xml()
    tallies.export_to_xml()

    print(f"\nRunning {ppm} ppm boron...")
    openmc.run(output=False)

    sp = openmc.StatePoint('statepoint.150.h5')
    k    = sp.keff.nominal_value
    kstd = sp.keff.std_dev
    keff_vals.append(k)
    keff_stds.append(kstd)
    print(f"  {ppm} ppm → k-eff = {k:.5f} ± {kstd:.5f}")

# ============================================================================
# INTERPOLATE CRITICAL BORON CONCENTRATION
# ============================================================================

keff_arr = np.array(keff_vals)
ppm_arr  = np.array(boron_ppm)

# Linear interpolation to find CBC (k=1.0)
cbc_interp = np.interp(1.0, keff_arr[::-1], ppm_arr[::-1])
print(f"\nPredicted Critical Boron Concentration: {cbc_interp:.0f} ppm")

# ============================================================================
# PLOT
# ============================================================================

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(ppm_arr, keff_arr, yerr=2*np.array(keff_stds),
            fmt='o-', capsize=4, color='steelblue', label='k-eff ± 2σ')
ax.axhline(1.0, color='red', linestyle='--', label='Critical (k=1.0)')
ax.axvline(cbc_interp, color='orange', linestyle='--',
           label=f'CBC ≈ {cbc_interp:.0f} ppm')
ax.set_xlabel('Boron Concentration (ppm)')
ax.set_ylabel('k-effective')
ax.set_title('AP1000 HZP BOC — Boron Letdown Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('boron_letdown.png', dpi=150)
plt.show()

print("\nBoron Letdown Summary:")
print(f"{'ppm':>8} {'k-eff':>10} {'±2σ':>10}")
print("-" * 30)
for ppm, k, s in zip(boron_ppm, keff_vals, keff_stds):
    print(f"{ppm:>8} {k:>10.5f} {2*s:>10.5f}")
print(f"\nPredicted CBC: {cbc_interp:.0f} ppm (no IFBA, no rods)")